# DQL: Aggregate Functions

Aggregate functions summarize many rows into one value or one value per group.


## Setup

Run this first. It creates a Spark session, finds the CSV files, loads them as DataFrames, and registers SQL temp views.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd(),
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or beside this notebook.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## COUNT

`count(*)` counts rows. `count(column)` counts non-null values in that column.


In [ ]:
emp.agg(
    F.count("*").alias("row_count"),
    F.count("commission").alias("employees_with_commission")
).show()

spark.sql("""
SELECT COUNT(*) AS row_count, COUNT(commission) AS employees_with_commission
FROM emp
""").show()


## SUM and AVG

Use `sum` for totals and `avg` for averages. `round` makes decimal output easier to read.


In [ ]:
emp.agg(
    F.sum("sal").alias("total_salary"),
    F.round(F.avg("sal"), 2).alias("avg_salary"),
    F.sum(F.coalesce("commission", F.lit(0))).alias("total_commission")
).show()

spark.sql("""
SELECT SUM(sal) AS total_salary,
       ROUND(AVG(sal), 2) AS avg_salary,
       SUM(COALESCE(commission, 0)) AS total_commission
FROM emp
""").show()


## MAX and MIN

Use `max` and `min` to find extremes.


In [ ]:
emp.agg(
    F.max("sal").alias("highest_salary"),
    F.min("sal").alias("lowest_salary"),
    F.min("hiredate").alias("first_hire_date"),
    F.max("hiredate").alias("last_hire_date")
).show()

spark.sql("""
SELECT MAX(sal) AS highest_salary,
       MIN(sal) AS lowest_salary,
       MIN(hiredate) AS first_hire_date,
       MAX(hiredate) AS last_hire_date
FROM emp
""").show()


## Aggregates by Group

Aggregates are often grouped by department, job, project, or salary grade.


In [ ]:
emp.join(dept, "deptno").groupBy("deptno", "dname").agg(
    F.count("*").alias("employee_count"),
    F.round(F.avg("sal"), 2).alias("avg_salary"),
    F.max("sal").alias("max_salary"),
    F.min("sal").alias("min_salary")
).orderBy("deptno").show()

spark.sql("""
SELECT d.deptno, d.dname,
       COUNT(*) AS employee_count,
       ROUND(AVG(e.sal), 2) AS avg_salary,
       MAX(e.sal) AS max_salary,
       MIN(e.sal) AS min_salary
FROM emp e
JOIN dept d ON e.deptno = d.deptno
GROUP BY d.deptno, d.dname
ORDER BY d.deptno
""").show()
